# Building a STAC catalog from scratch

**STAC** (SpatioTemporal Asset Catalog) is a common language for describing
geospatial data on the web.  Almost every modern EO data archive -- Landsat,
Sentinel, commercial providers -- exposes data through a STAC API.

The hierarchy is simple:

```
Catalog
  Collection  (a dataset, e.g. Sentinel-2 L2A)
    Item      (one scene, one granule)
      Asset   (an actual file: a band GeoTIFF, a thumbnail, ...)
```

In this notebook we:
1. Define metadata for two real Sentinel-2 scenes
2. Build the full STAC object tree with `pystac`
3. Validate everything against the STAC spec
4. Save it as static JSON files
5. Serve it on `localhost` and open it in STAC Browser


## 1. Install and import


In [1]:
%pip install -q 'pystac[validation]>=1.10' jsonschema


Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json, threading, functools, webbrowser

import pystac
from pystac import (
    Catalog, CatalogType,
    Collection, Extent, SpatialExtent, TemporalExtent,
    Item, Asset, MediaType, Link,
)

# Where the generated JSON files will live.
# Path is relative to this notebook, so '../catalog' lands one level up
# at lectures_demo/catalog/ regardless of where you cloned the repo.
CATALOG_DIR = Path('../catalog').resolve()
CATALOG_DIR.mkdir(parents=True, exist_ok=True)

# The port our local HTTP server will listen on.
# Change this if 8080 is already taken on your machine.
PORT = 8080

print(f'pystac {pystac.__version__}')
print(f'Catalog will be saved to: {CATALOG_DIR}')


pystac 1.14.3
Catalog will be saved to: /Users/michallupa/eolabs/lectures_demo/catalog


## Example Sentinel-2 scene metadata

Two summer scenes over Tuscany (UTM tile 32TNT).  Assets point to real
public Cloud-Optimised GeoTIFFs on the `sentinel-cogs` S3 bucket.
No download is needed -- the URLs are embedded as links in the STAC JSON.


In [3]:
# Root of the public Sentinel-2 COG archive on AWS.
# Every scene stored there follows the same URL pattern:
#   {BASE}/{utm_zone}/{grid_lat}/{grid_lon}/{year}/{month}/{scene_id}/{filename}
BASE = 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs'

# UTM tile 32TNT covers roughly the area around Florence, Italy.
# The four values are [west, south, east, north] in WGS84 degrees.
BBOX_32TNT = [9.76, 43.26, 11.52, 44.27]

def bbox_to_geom(bbox):
    """Turn a [west, south, east, north] bbox into a GeoJSON Polygon.
    STAC items require a proper GeoJSON geometry, not just a bbox array.
    The ring must close on itself, hence the repeated first point at the end.
    """
    w, s, e, n = bbox
    return {
        'type': 'Polygon',
        'coordinates': [[
            [w, s], [e, s], [e, n], [w, n], [w, s]  # SW -> SE -> NE -> NW -> SW
        ]]
    }

# Two real scenes acquired on different days by the two Sentinel-2 satellites.
# The scene IDs follow the ESA naming convention:
#   S2{sat}_{tile}_{date}_{processing_baseline}_{level}
# base_url points to the folder on S3 that holds all band GeoTIFFs for the scene.
SCENES = [
    {
        'id':          'S2A_32TNT_20240601_0_L2A',
        'datetime':    datetime(2024, 6, 1, 10, 16, 23, tzinfo=timezone.utc),
        'platform':    'sentinel-2a',   # the 'A' satellite of the constellation
        'cloud_cover': 3.2,             # percent cloud cover over the tile
        'bbox':        BBOX_32TNT,
        'base_url':    f'{BASE}/32/T/NT/2024/6/S2A_32TNT_20240601_0_L2A',
    },
    {
        'id':          'S2B_32TNT_20240612_0_L2A',
        'datetime':    datetime(2024, 6, 12, 10, 14, 55, tzinfo=timezone.utc),
        'platform':    'sentinel-2b',   # the 'B' satellite, slightly different orbit
        'cloud_cover': 8.7,
        'bbox':        BBOX_32TNT,
        'base_url':    f'{BASE}/32/T/NT/2024/6/S2B_32TNT_20240612_0_L2A',
    },
]

print(f'{len(SCENES)} scenes defined')
for s in SCENES:
    print(f"  {s['id']}  {s['datetime'].date()}  cloud={s['cloud_cover']}%")


2 scenes defined
  S2A_32TNT_20240601_0_L2A  2024-06-01  cloud=3.2%
  S2B_32TNT_20240612_0_L2A  2024-06-12  cloud=8.7%


## Build the STAC object tree

We work bottom-up: Assets go into Items, Items into a Collection,
the Collection into the Catalog.


In [4]:
# A Catalog is just a root container -- it has an id, a description,
# and links to its children (Collections or other Catalogs).
# Think of it as the index page of a data archive.
catalog = Catalog(
    id='sentinel-2-demo',
    description='Demo STAC catalog built with pystac. Two Sentinel-2 L2A scenes over Tuscany.',
    title='Sentinel-2 Demo',
)

# A Collection groups Items that share the same dataset definition --
# same sensor, same processing level, same coordinate system, etc.
# The Extent tells clients the overall spatial and temporal footprint
# without having to read every single Item.
spatial_extent = SpatialExtent(bboxes=[BBOX_32TNT])
temporal_extent = TemporalExtent(intervals=[
    # One interval: from the earliest to the latest scene datetime.
    # None as the second value would mean 'open-ended / ongoing'.
    [SCENES[0]['datetime'], SCENES[-1]['datetime']]
])

collection = Collection(
    id='sentinel-2-l2a',
    description='Sentinel-2 Level-2A surface reflectance (demo subset)',
    title='Sentinel-2 L2A',
    extent=Extent(spatial_extent, temporal_extent),
    license='proprietary',
    extra_fields={
        # These fields are not in the core STAC spec but are widely used
        # and understood by most STAC clients.
        'platform':      'sentinel-2',
        'constellation': 'sentinel-2',
        'instruments':   ['msi'],       # MultiSpectral Instrument
    },
)

# Each Item represents one scene at one point in time.
# Items hold the geometry, datetime, and all the actual file links (Assets).
for scene in SCENES:
    item = Item(
        id=scene['id'],
        geometry=bbox_to_geom(scene['bbox']),  # GeoJSON Polygon
        bbox=scene['bbox'],                     # duplicate as a simple array for quick filtering
        datetime=scene['datetime'],
        properties={
            # Standard EO extension fields -- any STAC-aware tool knows these.
            'platform':         scene['platform'],
            'constellation':    'sentinel-2',
            'instruments':      ['msi'],
            'eo:cloud_cover':   scene['cloud_cover'],
            'processing:level': 'L2A',
        },
    )

    base = scene['base_url']

    # Assets are the actual files. Each has a unique key within the Item,
    # a URL, a media type, a human-readable title, and optional roles.
    # Roles like 'thumbnail', 'visual', 'data' help clients decide
    # which asset to show or download without reading all of them.

    # Small JPEG preview -- useful for quick browsing in STAC Browser
    item.add_asset('thumbnail', Asset(
        href=f'{base}/thumbnail.jpg',
        media_type=MediaType.JPEG,
        title='Thumbnail',
        roles=['thumbnail'],
    ))

    # The TCI (True Colour Image) is a ready-made RGB composite at 10 m.
    # MediaType.COG signals that the file is a Cloud-Optimised GeoTIFF --
    # clients can stream just the pixels they need without a full download.
    item.add_asset('visual', Asset(
        href=f'{base}/TCI.tif',
        media_type=MediaType.COG,
        title='True colour (TCI, 10 m)',
        roles=['visual'],
    ))

    # Individual spectral bands. The eo:bands field carries wavelength
    # metadata that lets tools build spectral plots or calculate indices
    # without knowing the sensor-specific band naming.
    item.add_asset('red', Asset(
        href=f'{base}/B04.tif',
        media_type=MediaType.COG,
        title='Red - B04 (10 m)',
        roles=['data'],
        extra_fields={'eo:bands': [{
            'name': 'B04',
            'common_name': 'red',
            'center_wavelength': 0.665,   # micrometres
        }]},
    ))

    item.add_asset('nir', Asset(
        href=f'{base}/B08.tif',
        media_type=MediaType.COG,
        title='NIR - B08 (10 m)',
        roles=['data'],
        extra_fields={'eo:bands': [{
            'name': 'B08',
            'common_name': 'nir',
            'center_wavelength': 0.842,
        }]},
    ))

    # Wire the item into the collection, and the collection into the catalog.
    # pystac tracks these relationships as Link objects internally.
    collection.add_item(item)

catalog.add_child(collection)

print(f'Catalog   : {catalog.id}')
print(f'Collection: {collection.id}')
for item in collection.get_items():
    print(f'  Item: {item.id}  |  assets: {list(item.assets.keys())}')


Catalog : sentinel-2-demo
Collection: sentinel-2-l2a
  Item: S2A_32TNT_20240601_0_L2A  |  assets: ['thumbnail', 'visual', 'red', 'nir']
  Item: S2B_32TNT_20240612_0_L2A  |  assets: ['thumbnail', 'visual', 'red', 'nir']


## Save as static JSON

`normalize_hrefs` assigns a consistent file path to every object,
then `save` writes them out.  We validate **after** saving by reading
the files back from disk -- that checks exactly what gets served.


In [5]:
import shutil

# Wipe the output folder so a re-run always produces a clean result.
# Without this, old item folders from a previous run would linger.
if CATALOG_DIR.exists():
    shutil.rmtree(CATALOG_DIR)
CATALOG_DIR.mkdir(parents=True)

# normalize_hrefs walks the entire object tree and assigns a local file path
# to every Catalog, Collection, and Item based on the root directory we give it.
# The resulting structure mirrors the hierarchy:
#   catalog/catalog.json
#   catalog/sentinel-2-l2a/collection.json
#   catalog/sentinel-2-l2a/S2A_.../S2A_....json
catalog.normalize_hrefs(str(CATALOG_DIR))

# SELF_CONTAINED means every JSON file uses absolute hrefs for its links,
# so the catalog works the same whether you serve it from localhost,
# an S3 bucket, or a GitHub Pages site -- no path-relative guessing needed.
catalog.save(catalog_type=CatalogType.SELF_CONTAINED)

files = sorted(CATALOG_DIR.rglob('*.json'))
print(f'Written {len(files)} JSON files:')
for f in files:
    print(f'  {f.relative_to(CATALOG_DIR)}')


Written 4 JSON files:
  catalog.json
  sentinel-2-l2a/S2A_32TNT_20240601_0_L2A/S2A_32TNT_20240601_0_L2A.json
  sentinel-2-l2a/S2B_32TNT_20240612_0_L2A/S2B_32TNT_20240612_0_L2A.json
  sentinel-2-l2a/collection.json


## Validate against the STAC spec

Reading back from disk ensures we validate the exact JSON that will be served,
including all resolved links.  `pystac.read_file` reconstructs the full object
from the saved file so the validator sees proper `href` strings everywhere.


In [6]:
# We validate the saved files rather than the in-memory objects.
# pystac.read_file re-reads each JSON from disk, which means all links
# are fully resolved hrefs -- no None values that could trip up the schema check.
print('Validating saved files...')
for json_file in sorted(CATALOG_DIR.rglob('*.json')):
    obj = pystac.read_file(str(json_file))
    obj.validate()   # raises STACValidationError if something is wrong
    label = json_file.relative_to(CATALOG_DIR)
    print(f'  {label}: OK')

print('\nAll objects valid.')

# Show the root catalog JSON so you can see what a client would receive first.
# The 'links' array is the key part -- it tells the client where to find
# the child collections.
print('\n--- catalog.json ---')
root = json.loads((CATALOG_DIR / 'catalog.json').read_text())
print(json.dumps(root, indent=2))


Validating saved files...
  catalog.json: OK
  sentinel-2-l2a/S2A_32TNT_20240601_0_L2A/S2A_32TNT_20240601_0_L2A.json: OK
  sentinel-2-l2a/S2B_32TNT_20240612_0_L2A/S2B_32TNT_20240612_0_L2A.json: OK
  sentinel-2-l2a/collection.json: OK

All objects valid.

--- catalog.json ---
{
  "type": "Catalog",
  "id": "sentinel-2-demo",
  "stac_version": "1.1.0",
  "description": "Demo STAC catalog built with pystac. Two Sentinel-2 L2A scenes over Tuscany.",
  "links": [
    {
      "rel": "root",
      "href": "./catalog.json",
      "type": "application/json",
      "title": "Sentinel-2 Demo"
    },
    {
      "rel": "child",
      "href": "./sentinel-2-l2a/collection.json",
      "type": "application/json",
      "title": "Sentinel-2 L2A"
    }
  ],
  "title": "Sentinel-2 Demo"
}


## Serve the catalog on localhost

A minimal HTTP server with CORS headers so that the hosted STAC Browser
(which runs on `https://`) can fetch from your `http://localhost`.
The server runs in a background thread and stays alive for the
duration of the kernel session.


In [7]:
from http.server import HTTPServer, SimpleHTTPRequestHandler

class CORSHandler(SimpleHTTPRequestHandler):
    """A minimal HTTP handler that injects CORS headers into every response.

    Without Access-Control-Allow-Origin: * the hosted STAC Browser (which
    runs on https://radiantearth.github.io) would be blocked by the browser
    from fetching our http://localhost responses -- that's the browser's
    same-origin policy in action.
    """
    def end_headers(self):
        self.send_header('Access-Control-Allow-Origin',  '*')
        self.send_header('Access-Control-Allow-Headers', '*')
        # Tell the browser not to cache responses so edits show up immediately.
        self.send_header('Cache-Control', 'no-store')
        super().end_headers()

    def log_message(self, *args):
        pass  # suppress the per-request console noise

# functools.partial pre-fills the 'directory' argument so the handler
# serves files from our catalog folder, not from the notebook folder.
handler = functools.partial(CORSHandler, directory=str(CATALOG_DIR))

try:
    server = HTTPServer(('localhost', PORT), handler)
    # daemon=True means the thread shuts down automatically when the
    # Jupyter kernel exits -- no need to manually stop it.
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    print(f'Server running on http://localhost:{PORT}/')
except OSError:
    # This happens if you run the cell twice without restarting the kernel.
    # The previous server is still bound to the port, which is fine.
    print(f'Port {PORT} already in use -- server is already running.')

# Quick sanity check: fetch the catalog root and confirm it responds.
import urllib.request
try:
    with urllib.request.urlopen(f'http://localhost:{PORT}/catalog.json') as r:
        data = json.loads(r.read())
    print(f'Catalog root reachable: id = "{data["id"]}"')
except Exception as e:
    print(f'Could not reach catalog: {e}')


Server running on http://localhost:8080/
Catalog root reachable: id = "sentinel-2-demo"


## Open in STAC Browser

The Radiant Earth STAC Browser is a hosted web app that can point at any
STAC catalog URL -- including your localhost one.

> If your browser blocks mixed content (HTTPS page fetching HTTP),
> open the URL manually and allow it, or use Firefox which prompts you.


In [8]:
ROOT_URL    = f'http://localhost:{PORT}/catalog.json'

# The Radiant Earth STAC Browser is a hosted single-page app.
# The '#/external/' fragment tells it to load a catalog from an external URL
# instead of a built-in example -- that's how we point it at localhost.
BROWSER_URL = f'https://radiantearth.github.io/stac-browser/#/external/{ROOT_URL}'

print('Catalog root :', ROOT_URL)
print('STAC Browser :', BROWSER_URL)

# Open in whatever browser is set as default on this machine.
webbrowser.open(BROWSER_URL)

# Also render it as an iframe in the notebook output.
# Note: some browsers block mixed content (HTTPS page loading HTTP assets).
# If the iframe shows blank, open the STAC Browser URL directly in a tab.
from IPython.display import IFrame, display
display(IFrame(BROWSER_URL, width='100%', height=600))


Catalog root : http://localhost:8080/catalog.json
STAC Browser : https://radiantearth.github.io/stac-browser/#/external/http://localhost:8080/catalog.json


## Explore the raw JSON

You can also inspect each file directly.  Here we pretty-print one Item
to see exactly what STAC JSON looks like on disk.


In [9]:
# Read one of the saved Item JSON files so you can inspect the full structure.
# This is exactly what a STAC client (or the Browser) fetches when navigating
# from the Collection down to an individual scene.
item_files = sorted(CATALOG_DIR.rglob('*.json'))
item_files = [f for f in item_files if 'S2' in f.name]  # skip collection.json / catalog.json

if item_files:
    item_json = json.loads(item_files[0].read_text())
    print(f'--- {item_files[0].name} ---')
    # default=str handles datetime objects that json.dumps can't serialise natively
    print(json.dumps(item_json, indent=2, default=str))


--- S2A_32TNT_20240601_0_L2A.json ---
{
  "type": "Feature",
  "stac_version": "1.1.0",
  "stac_extensions": [],
  "id": "S2A_32TNT_20240601_0_L2A",
  "geometry": {
    "type": "Polygon",
    "coordinates": [
      [
        [
          9.76,
          43.26
        ],
        [
          11.52,
          43.26
        ],
        [
          11.52,
          44.27
        ],
        [
          9.76,
          44.27
        ],
        [
          9.76,
          43.26
        ]
      ]
    ]
  },
  "bbox": [
    9.76,
    43.26,
    11.52,
    44.27
  ],
  "properties": {
    "platform": "sentinel-2a",
    "constellation": "sentinel-2",
    "instruments": [
      "msi"
    ],
    "eo:cloud_cover": 3.2,
    "processing:level": "L2A",
    "datetime": "2024-06-01T10:16:23Z"
  },
  "links": [
    {
      "rel": "root",
      "href": "../../catalog.json",
      "type": "application/json",
      "title": "Sentinel-2 Demo"
    },
    {
      "rel": "collection",
      "href": "../collection.j